#Building a GPT model by following Karpathy's "Let's build a GPT" tutorial

this is a longer,(and admittedly messier) version i wrote to understand language models better. I believe it will be beneficial for people not as proficient in python as me.



In [1]:
#importing text
with open('data/input.txt', 'r', encoding='utf-8') as f:
    text=f.read()

print(len(text))

1115394


In [2]:
#sorting all 'characters' in the file
charss = sorted(list(set(text)))
vocab_size = len(charss)
 #we need to tokenize (convert these into integers)

In [3]:
string_to_integer = {} #defining a dictionary
for i in range(len(charss)):
    ch = charss[i] #assigning each character an integer
    string_to_integer[ch] = i #character is your key and the value is your position
#vice-versa
integer_to_string = {}
for i in range(len(string_to_integer)):
    integer_to_string[i] = charss[i]





In [4]:
#defining the encode and decode functions
def encode(s):
    result =[]
    for i in s: #goes through each character of the string
        result.append(string_to_integer[i])
    return result

def decode(i):
    result = []
    for c in i:
        result.append(integer_to_string[c])
    return ''.join(result) #joins all the characters into a string and returns it

    #testing
print(encode("to be or not to be"))
print(decode(encode("to be or not to be")))




[58, 53, 1, 40, 43, 1, 53, 56, 1, 52, 53, 58, 1, 58, 53, 1, 40, 43]
to be or not to be


As we can see the text has been tokenized(broken down into its characters) into individual numbers, (vectors) and is encoding/decoding correctly.

there are other ways to 'tokenize' the text, (look up tiktoken) but this is the easiest way to get started

we are trading off sequence length for a smaller 'vocabulary' size

In [5]:
#we are now going to tokenize the entire data.

import torch #importing pytorch (very important library for ml)
data = torch.tensor(encode(text), dtype = torch.long) #storing the encoded text as a tensor in pytorch 
#a tensor is an array, but unlike numpy arrays it can remember how its been changed and run on GPUs
#dtype is just defining the datatype.

#print(data.shape,data.dtype)
#print(data[:1000]) #prints the encoding for the first 1000 characters

In [6]:
#going on to seperate our data into a 'training' section and a 'validating' section
n = int(0.9 * len(data)) #first 90% of the data will be used to train our model
train_data = data[:n]
val_data = data[n:]

In [7]:
#we have to use chunks, dont process entire data

block_size = 8 #context length
train_data[:block_size+1]
train_data[:block_size+1].shape



torch.Size([9])

In [8]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for i in range (block_size):
    context = x[:i+1]
    target = y[i]
    print(f"when input is {context} the target is {target}")

#train using next token

when input is tensor([18]) the target is 47
when input is tensor([18, 47]) the target is 56
when input is tensor([18, 47, 56]) the target is 57
when input is tensor([18, 47, 56, 57]) the target is 58
when input is tensor([18, 47, 56, 57, 58]) the target is 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


In [9]:
batch_size = 4 #no. of chunks
block_size = 8# size of chunks

def get_batch(split):
    data = train_data if split == 'train' else val_data
    indices = torch.randint(len(data)-block_size, (batch_size,)) #first input defines max value, (subtract block size since we cant go outside the limit)
#batch size, we need 4 examples.
    print(indices)#prints the randomly selected index from the data file.
    
    x = torch.stack([data[i:i+block_size]for i in indices]) 
    y = torch.stack([data[i+1:i+1+block_size]for i in indices]) 

    #torch.stack, 'stacks' all the tensors which we obtain after iterating through indices in a 2D tensor
    return x,y


xb,yb = get_batch('train')
print('inputs')
print(xb.shape)#context
print(xb)
print('targets')
print(yb.shape)#targets
print(yb)

for b in range(batch_size):
    for bl in range(block_size):
        context = xb[b, :bl + 1]
        target = yb[b,bl]
        print(f"when input is {context.tolist()} the target {target}")


tensor([319435, 638179, 671108,  40022])
inputs
torch.Size([4, 8])
tensor([[ 0, 18, 47, 56, 57, 58,  6,  1],
        [43, 52, 45, 58, 46, 43, 52,  1],
        [ 0, 37, 43, 58,  6,  1, 35, 39],
        [46, 53, 51, 43,  1, 44, 53, 56]])
targets
torch.Size([4, 8])
tensor([[18, 47, 56, 57, 58,  6,  1, 58],
        [52, 45, 58, 46, 43, 52,  1, 39],
        [37, 43, 58,  6,  1, 35, 39, 56],
        [53, 51, 43,  1, 44, 53, 56,  1]])
when input is [0] the target 18
when input is [0, 18] the target 47
when input is [0, 18, 47] the target 56
when input is [0, 18, 47, 56] the target 57
when input is [0, 18, 47, 56, 57] the target 58
when input is [0, 18, 47, 56, 57, 58] the target 6
when input is [0, 18, 47, 56, 57, 58, 6] the target 1
when input is [0, 18, 47, 56, 57, 58, 6, 1] the target 58
when input is [43] the target 52
when input is [43, 52] the target 45
when input is [43, 52, 45] the target 58
when input is [43, 52, 45, 58] the target 46
when input is [43, 52, 45, 58, 46] the target 43


In [10]:
print(xb)#our input to the transformer

tensor([[ 0, 18, 47, 56, 57, 58,  6,  1],
        [43, 52, 45, 58, 46, 43, 52,  1],
        [ 0, 37, 43, 58,  6,  1, 35, 39],
        [46, 53, 51, 43,  1, 44, 53, 56]])


starting off with the bigram language model.
A bigram model predicts the next character by looking only at the current character: for each of the 65 characters it learns a table of scores for which character usually comes next. It's the simplest possible language model, and it gives you a baseline to beat, plus the full pipeline (data → model → loss → training → generating text) before attention gets added.


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F #holds plain functions with no weights
#logits are our list of raw unnormalized scores of the succeeding token (65 tokens - 65 scores - highest prob gets placed in bigram)
class BigramLangModel(nn.Module):
    def __init__(self,vocab_size): 
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size) #converts our tensor values by giving it rows of logits


    def forward(self,inputs, targets=None):
        if(targets == None):
            loss = None
        else:

            logits = self.token_embedding_table(inputs) #creating logits (randomized at first) #(BATCH,TIME,CHANNEL) TENSOR
            B,T,C = logits.shape #unpacking 4,8,65 into B T AND C
            logits = logits.view(B*T,C) #changing arrangement of array to 32*65 #created a 2D array
            #every single 'token' from our input will pluck out its row from the embedding table


            #now targets are in shape (b,t)2D ARRAY  i want to turn it into a 1D array of bt rows

            targets = targets.view(B*T)#4,8 to 32
            #we have fit logits and targets as per pytorch doc.

            loss = F.cross_entropy(logits,targets) #gives ur loss woohoo #cross entropy (simplified) = -log(probability) thats why initial loss is around -log(1/65)

        return logits,loss

    def generate(self,idx,max_token_value): #max_token_value = numbers
        for i in range(max_token_value):
            logits,loss = self(idx) #runs forward, returns just logits
            logits = logits[:,-1,:] #taking last digit of T ,shape is (B,C)
            probs = F.softmax(logits, dim =-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx,idx_next), dim =1) #(B,T+1)

        return idx


m = BigramLangModel(vocab_size) #initializes a 65*65 embedding map
logits,loss = m(xb,yb) #nothing is being done with our yb as of now.
print(logits.shape) #4,8,65 our batch size, chunk size and our logit row size
print(loss)
#print(out)
#each element is substituted by its scores.

#would highly recommend one watches https://youtu.be/Qd2bAzwH9uA?si=n9XB1NDXkyzciGXD before implementing this code for conceptual clarity

#pytorch accepts (B,C,T) but your tensor is in (B,T,C) so you cant directly pass it ( you have (4,8,65) but pytorch wants (4,65,8)) - check pytorch cross entropy documentation

#great llm explanation - "The model sees L and outputs 65 logits, one for each possible next character. Softmax turns them into 65 probabilities that add up to 1. The target (43, which is e) tells you which one of those 65 probabilities to look at. If the model gave e a probability of 0.02, the loss for that prediction is −ln(0.02) ≈ 3.9, and the other 64 probabilities are ignored. The batch loss is the average of these penalties over all 32 predictions."





torch.Size([32, 65])
tensor(4.4082, grad_fn=<NllLossBackward0>)
